In [ ]:
!pip install lightning wandb peft

In [4]:
import pandas as pd
import numpy as np
from lightning.pytorch.callbacks import ModelCheckpoint

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Code

In [5]:
!pip install lightning wandb peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 33.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 35.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.8/868.8 kB 42.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.3/802.3 kB 42.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 30.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.2/289.2 kB 36.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 31.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 10.7 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB

In [6]:
import torch
from lightning.pytorch import LightningDataModule

class my_dataset(torch.utils.data.Dataset):
  def __init__(self, tokenizer, speech_data, label_data):
    self.tokenizer = tokenizer
    self.tokenizer.pad_token = tokenizer.eos_token
    self.speech = speech_data
    self.label = label_data
    self.label_to_id = {'R': [0], 'D': [1]}

  def __len__(self):
    return len(self.speech)

  def __getitem__(self, index):
    text = self.tokenizer.bos_token + self.speech['speech'][index] + self.tokenizer.eos_token
    input_ids = self.tokenizer.encode(text, return_tensors='pt', padding='max_length', truncation=True, max_length=512)[0]
    pad_id = 2
    attention_mask = torch.Tensor([1 if el != pad_id else 0 for el in input_ids]).to(dtype=torch.long)

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'label': self.label_to_id [self.label['party'][index]]
    }

class DataModule(LightningDataModule):
    def __init__(
            self,
            model_type,
            batch_size,
            train_speech_df,
            train_label_df,
            valid_speech_df,
            valid_label_df,
    ):
        super().__init__()
        self.batch_size = batch_size
        self.train_speech_df = train_speech_df
        self.train_label_df = train_label_df
        self.valid_speech_df = valid_speech_df
        self.valid_label_df = valid_label_df
        self.tokenizer = AutoTokenizer.from_pretrained(model_type)

    def prepare_data(self):
      pass

    def setup(self, stage: str):
        self.train_ds = my_dataset(self.tokenizer, self.train_speech_df, self.train_label_df)
        self.valid_ds = my_dataset(self.tokenizer, self.valid_speech_df, self.valid_label_df)


    def train_dataloader(self):
        loader = torch.utils.data.DataLoader(self.train_ds, batch_size=self.batch_size, drop_last=True, pin_memory=True)
        return loader

    def val_dataloader(self):
        loader = torch.utils.data.DataLoader(self.valid_ds, batch_size=self.batch_size, drop_last=True, pin_memory=True)
        return loader



In [ ]:
import time
from lightning.pytorch.callbacks import ModelCheckpoint

wandb.login()

model_name="microsoft/mdeberta-v3-base"
batch_size=16

#Abortion training (without finetuning on Iraq)
deberta_module_abortion_initial = DebertaModule(use_lora=True)
timestamp = time.strftime("%Y%m%d-%H%M%S")

checkpoint_callback_abortion_initial = ModelCheckpoint(
    monitor='val_loss',
    dirpath=checkpoint_dir,
    filename=f'deberta-initial-{timestamp}-{{epoch:02d}}-{{val_loss:.2f}}',
    save_top_k=1,
    mode='min'
)

trainer_abortion_initial = L.Trainer(
    devices=1,
    accelerator="gpu",
    max_epochs=10,
    num_sanity_val_steps=0,
    logger=WandbLogger(log_model="all", project='Deberta_initial_abortion'),
    callbacks=[checkpoint_callback_abortion_initial]
)

# trainer_abortion_initial.fit(model=deberta_module_abortion_initial, datamodule=DataModule(model_name, batch_size, train_abortion_df, train_abortion_label_df, test_abortion_df, test_abortion_label_df))

# # write name of the path here for saved model
# initial_best_model_path = checkpoint_callback_abortion_initial.best_model_path

# deberta_module_eval = DebertaModule.load_from_checkpoint(initial_best_model_path, use_lora=True)
# initial_results = trainer_abortion_initial.validate(model=deberta_module_eval, datamodule=DataModule(model_name, batch_size, None, None, test_data, test_labels))
# print(initial_results)

# wandb.finish()

In [7]:
import os
from torch import optim, nn, utils, Tensor
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
import lightning as L

from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from peft import get_peft_config, get_peft_model, PeftModel, PeftConfig, LoraConfig, TaskType
import torch.nn.functional as F

from torch.optim.lr_scheduler import CosineAnnealingLR

import wandb
from lightning.pytorch.loggers import WandbLogger

# define the LightningModule
class DebertaModule(L.LightningModule):
    def __init__(self, use_lora=False):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained("microsoft/mdeberta-v3-base")
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModel.from_pretrained("microsoft/mdeberta-v3-base")
        self.proj = nn.Linear(768, 1)
        total_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad) + sum(p.numel() for p in self.proj.parameters() if p.requires_grad)
        print(f"\n--> {total_params / 1e6} Million traininable params before LORA\n")
        if use_lora:
          peft_config = LoraConfig(
              target_modules=['query_proj', 'value_proj'],
              r=8,
              lora_alpha=32,
              lora_dropout=0.05
          )
          self.model = get_peft_model(self.model, peft_config)
          total_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad) + sum(p.numel() for p in self.proj.parameters() if p.requires_grad)
          print(f"\n--> {total_params / 1e6} Million trainable params after LORA\n")

        deberta_dim = 768

        self.loss_func = F.binary_cross_entropy_with_logits
        self.valid_preds = []
        self.valid_labels = []


    def training_step(self, batch, batch_idx):
        #labels = torch.stack(batch['label']).T.to(dtype=torch.float16)
        labels = torch.stack(batch['label']).T.to(dtype=torch.float16).view(-1, 1)
        del batch['label']
        deberta_logits = self.model(**batch).last_hidden_state[:, 0]
        logits = self.proj(deberta_logits)
        loss = self.loss_func(logits, labels)
        self.log_dict({'loss': loss}, prog_bar=True)

        lr = self.trainer.optimizers[0].param_groups[0]['lr']
        self.log('lr', lr, prog_bar=True)

        return {'loss': loss}

    def validation_step(self, batch, batch_idx):
        #labels = torch.stack(batch['label'])
        labels = torch.stack(batch['label']).T.to(dtype=torch.float16).view(-1, 1)
        del batch['label']
        deberta_logits = self.model(**batch).last_hidden_state[:, 0]
        logits = self.proj(deberta_logits)
        val_loss = self.loss_func(logits, labels.to(dtype=torch.float16))
        preds = torch.round(F.sigmoid(logits))
        self.valid_preds.append(preds)
        self.valid_labels.append(labels)
        self.log('val_loss', val_loss, prog_bar=True)
        return {'val_loss': val_loss}

    def on_validation_epoch_end(self):
        TP, TN, FP, FN = 0, 0, 0, 0
        valid_preds = torch.stack(self.valid_preds).view(-1)
        valid_labels = torch.stack(self.valid_labels).view(-1)
        for pred, gold in zip(valid_preds, valid_labels):
          if pred == 1:
            if gold == 1:
              TN += 1
            else:
              FP += 1
          else:
            if gold == 1:
              FN += 1
            else:
              TN += 1

        recall = TN / (TN + FP)
        precision = TN / (TN + FN)
        acc = (TN + TP) / (TN + FP + FN + TP)
        f1 = 2 * recall * precision / (recall + precision)
        print('Recall', recall)
        print('Precision', precision)
        print('F1', f1)
        print('Accuracy', acc)
        self.log_dict({'Recall': recall,
                       'Accuracy': acc,
                       'Precision': precision,
                       'F1': f1,
                       }, prog_bar=True)
        self.valid_preds.clear()
        self.valid_labels.clear()


    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=3e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=10, eta_min=1e-6)
        sch_config = {
            "scheduler": scheduler,
            "interval": "step"
        }
        return {"optimizer": optimizer, "lr_scheduler": sch_config}

    def get_progress_bar_dict(self):
        items = super().get_progress_bar_dict()
        items.pop("v_num", None)
        return items

# Get files

In [10]:
test_data_dir = "/content/drive/MyDrive/CS 224N Project/Evaluation/Data/Processed_data/Test/"

In [8]:
checkpoint_dir = '/content/drive/MyDrive/ModelCheckpoints/'

In [11]:
def get_test_data(file_name, data_dir=test_data_dir):
  data = pd.read_csv(data_dir + file_name, index_col=None)
  np.shape(data)
  return data


In [12]:

test_data = get_test_data('main_iraq_data.csv')
test_labels = get_test_data('main_iraq_labels.csv')

# Get model

In [9]:
best_model_path = checkpoint_dir + 'abortion-finetune.ckpt'

deberta_module_eval = DebertaModule.load_from_checkpoint(best_model_path, use_lora=True)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/convert_slow_tokenizer.py:560: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]


--> 278.219521 Million traininable params before LORA


--> 0.295681 Million trainable params after LORA



In [ ]:
checkpoint_callback_finetune = ModelCheckpoint(
          monitor='val_loss',
          dirpath=checkpoint_dir,
          filename=f'deberta-finetune-{timestamp}-{{epoch:02d}}-{{val_loss:.2f}}',
          save_top_k=1,
          mode='min'
      )

In [ ]:
use_lora=True
model_name="microsoft/mdeberta-v3-base"
batch_size=16

timestamp = time.strftime("%Y%m%d-%H%M%S")

checkpoint_dir = '/content/drive/MyDrive/ModelCheckpoints'


initial_best_model_path = '/content/drive/MyDrive/ModelCheckpoints/abortion-no-finetune'
deberta_module_finetune = DebertaModule.load_from_checkpoint(initial_best_model_path, use_lora=use_lora)
checkpoint_callback_finetune = ModelCheckpoint(
          monitor='val_loss',
          dirpath=checkpoint_dir,
          filename=f'deberta-finetune-{timestamp}-{{epoch:02d}}-{{val_loss:.2f}}',
          save_top_k=1,
          mode='min'
      )

trainer_finetune = L.Trainer(
          devices=1,
          accelerator="gpu",
          max_epochs=10,
          num_sanity_val_steps=0,
          logger=WandbLogger(log_model="all", project='Deberta_finetune'),
          callbacks=[checkpoint_callback_finetune]
)

trainer_finetune.fit(model=deberta_module_finetune, datamodule=DataModule(model_name, batch_size, train_main_iraq, train_main_iraq_label, val_main_iraq, val_main_iraq_label))
finetune_results = trainer_finetune.validate(model=deberta_module_finetune, datamodule=DataModule(model_name, batch_size, None, None, test_data, test_labels))
print(finetune_results)
wandb.finish()

In [ ]:
test_data = test_main_iraq_df
test_labels = test_main_iraq_label_df